In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import t


## Settings

In [ ]:
# Set display options for pandas
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

In [ ]:
# Define dataset variable configurations
DATASET_VARS = {
    "BH_1": {
        "obj_var": "yield",
        "cat_vars": [
            "Aryl_halide_SMILES",
            "Additive_SMILES",
            "Base_SMILES",
            "Ligand_SMILES",
        ],
        "con_vars": [],
    },
    "DA": {
        "obj_var": "yield",
        "cat_vars": ["Base_SMILES", "Ligand_SMILES", "Solvent_SMILES"],
        "con_vars": ["Concentration", "Temp_C"],
    },
    "alkox": {
        "obj_var": "conversion",
        "cat_vars": [],
        "con_vars": ["catalase", "peroxidase", "alcohol_oxidase", "ph"],
    },
    "oer_plate_a": {
        "obj_var": "overpotential",
        "cat_vars": [],
        "con_vars": ["ni_load", "fe_load", "co_load", "mn_load", "ce_load", "la_load"],
    },
    "p3ht": {
        "obj_var": "conductivity",
        "cat_vars": [],
        "con_vars": [
            "p3ht_content",
            "d1_content",
            "d2_content",
            "d6_content",
            "d8_content",
        ],
    },
    "photo_pce10": {
        "obj_var": "degradation",
        "cat_vars": [],
        "con_vars": ["mat_1", "mat_2", "mat_3", "mat_4"],
    },
    "photo_wf3": {
        "obj_var": "degradation",
        "cat_vars": [],
        "con_vars": ["mat_1", "mat_2", "mat_3", "mat_4"],
    },
    "suzuki_edbo": {
        "obj_var": "yield",
        "cat_vars": ["electrophile", "nucleophile", "base", "ligand", "solvent"],
        "con_vars": [],
    },
    "suzuki": {
        "obj_var": "yield",
        "cat_vars": [],
        "con_vars": ["temperature", "pd_mol", "arbpin", "k3po4"],
    },
}

In [ ]:
DATASET_NAME_ORDER = [
    "BH_1",
    "DA",
    "alkox",
    "oer_plate_a",
    "p3ht",
    "photo_pce10",
    "photo_wf3",
    "suzuki_edbo",
    "suzuki",
]
MODEL_ORDER = [
    "gpt-5-mini-2025-08-07",
    "o4-mini-2025-04-16",
    "gpt-4.1-mini-2025-04-14",
    "gpt-4o-mini-2024-07-18",
    "claude-sonnet-4-5-20250929",
    "claude-haiku-4-5-20251001",
    "claude-3-5-haiku-20241022",
]
MODEL_LABELS = {
    "gpt-5-mini-2025-08-07": "GPT-5 mini",
    "o4-mini-2025-04-16": "o4 mini",
    "gpt-4.1-mini-2025-04-14": "GPT-4.1 mini",
    "gpt-4o-mini-2024-07-18": "GPT-4o mini",
    "claude-sonnet-4-5-20250929": "Claude Sonnet 4.5",
    "claude-haiku-4-5-20251001": "Claude Haiku 4.5",
    "claude-3-5-haiku-20241022": "Claude 3.5 Haiku",
}

In [ ]:
TIME_TAGS = [
    "20251109144312",
    "20251109144344",
    "20251109144638",
    "20251109144719",
    "20251109144852",
    "20251109144939",
]

In [ ]:
REPEAT_NUM = 5

In [ ]:
# Read and concatenate batch output logs
batch_output_logs_df_list = []
for time_tag in TIME_TAGS:
    batch_output_logs_df_tmp = pd.read_csv(
        Path("./results_final") / f"results_{time_tag}" / "batch_output_logs.csv"
    )
    batch_output_logs_df_tmp.insert(2, "time_tag", time_tag)
    batch_output_logs_df_list.append(batch_output_logs_df_tmp)
batch_output_logs_df = pd.concat(batch_output_logs_df_list, axis=0, ignore_index=True)

# Sort by dataset_name and model
batch_output_logs_df["dataset_name"] = pd.Categorical(
    batch_output_logs_df["dataset_name"], categories=DATASET_NAME_ORDER, ordered=True
)
batch_output_logs_df["model"] = pd.Categorical(
    batch_output_logs_df["model"], categories=MODEL_ORDER, ordered=True
)
batch_output_logs_df = batch_output_logs_df.sort_values(
    ["dataset_name", "model"]
).reset_index(drop=True)

# Convert to string type
batch_output_logs_df["dataset_name"] = batch_output_logs_df["dataset_name"].astype(str)
batch_output_logs_df["model"] = batch_output_logs_df["model"].astype(str)

# Create a new column containing model parameters
batch_output_logs_df["model_w_params"] = batch_output_logs_df["model"].map(MODEL_LABELS)
for idx, row in batch_output_logs_df.iterrows():
    if pd.notna(batch_output_logs_df.loc[idx, "reasoning_effort"]):
        batch_output_logs_df.loc[idx, "model_w_params"] += (
            " " + batch_output_logs_df.loc[idx, "reasoning_effort"]
        )
    elif pd.notna(batch_output_logs_df.loc[idx, "temperature"]):
        batch_output_logs_df.loc[idx, "model_w_params"] += " temp" + str(
            round(batch_output_logs_df.loc[idx, "temperature"], 2)
        )

# Save the summarized batch output logs
batch_output_logs_df.to_csv("results_final/batch_output_logs.csv", index=False)

## Bias, Preference

In [ ]:
batch_output_logs_df_wo_duplicate = (
    batch_output_logs_df[["dataset_name", "model", "model_w_params", "time_tag"]]
    .drop_duplicates()
    .reset_index(drop=True)
)
display(batch_output_logs_df_wo_duplicate)

In [ ]:
pair_result_dfs = {}

for i, batch_output_logs_df_row in batch_output_logs_df_wo_duplicate[
    ["dataset_name", "model", "model_w_params", "time_tag"]
].iterrows():
    dataset_name = batch_output_logs_df_row.dataset_name
    model = batch_output_logs_df_row.model
    model_w_params = batch_output_logs_df_row.model_w_params
    time_tag = batch_output_logs_df_row.time_tag

    print(f"dataset: {dataset_name}, model {model_w_params}")

    # Read experimental data
    exp_original_df = pd.read_csv(
        f"./dataset_processed/dataset_{dataset_name}_extracted.csv"
    )

    # Read original pair data
    pair_original_df = pd.read_csv(
        f"./dataset_processed/dataset_{dataset_name}_pair.csv"
    )
    obj_var = DATASET_VARS[dataset_name]["obj_var"]

    # Align judgment with LLM
    pair_original_df["setup_true"] = ""
    if dataset_name in [
        "oer_plate_a",
        "photo_pce10",
        "photo_wf3",
    ]:  # Lower is better
        pair_original_df.loc[
            pair_original_df[f"{obj_var}_A"] > pair_original_df[f"{obj_var}_B"],
            "setup_true",
        ] = "B"
        pair_original_df.loc[
            pair_original_df[f"{obj_var}_A"] < pair_original_df[f"{obj_var}_B"],
            "setup_true",
        ] = "A"
    else:
        pair_original_df.loc[
            pair_original_df[f"{obj_var}_A"] > pair_original_df[f"{obj_var}_B"],
            "setup_true",
        ] = "A"
        pair_original_df.loc[
            pair_original_df[f"{obj_var}_A"] < pair_original_df[f"{obj_var}_B"],
            "setup_true",
        ] = "B"
    assert len(pair_original_df.loc[pair_original_df["setup_true"] == ""]) == 0

    # Add features
    pair_original_df[f"{obj_var}_diff"] = (
        pair_original_df[f"{obj_var}_A"] - pair_original_df[f"{obj_var}_B"]
    )
    pair_original_df[f"{obj_var}_diff_abs"] = (
        pair_original_df[f"{obj_var}_A"] - pair_original_df[f"{obj_var}_B"]
    ).abs()

    # Read pair prediction results
    pair_result_merged_df = pair_original_df.copy()
    repeat_forward_cols = []
    repeat_reverse_cols = []
    for repeat in range(REPEAT_NUM):
        for forward_or_reverse in ["forward", "reverse"]:
            pair_result_df = pd.read_csv(
                Path("./results_final")
                / f"results_{time_tag}"
                / f"dataset_{dataset_name}_{model}_rep{repeat + 1}_{forward_or_reverse}_pair_result.csv"
            )
            setup_renamed = f"setup_predicted_{forward_or_reverse}_{repeat + 1}"
            pair_result_df = pair_result_df.rename(columns={"setup": setup_renamed})
            pair_result_df = pair_result_df[["ID_A", "ID_B", setup_renamed]]

            # Merge results (reverse pair prediction results swap ID_A and ID_B columns for merging)
            if forward_or_reverse == "reverse":
                pair_result_df = pair_result_df.rename(
                    columns={"ID_A": "ID_B", "ID_B": "ID_A"}
                )
            pair_result_merged_df = pd.merge(
                pair_result_merged_df, pair_result_df, on=["ID_A", "ID_B"], how="left"
            )

            if forward_or_reverse == "forward":
                repeat_forward_cols.append(setup_renamed)
            else:
                repeat_reverse_cols.append(setup_renamed)

    # Remove rows with NaN values
    pair_result_merged_df = pair_result_merged_df.dropna(
        subset=repeat_forward_cols + repeat_reverse_cols, how="any", axis=0
    )

    # Accuracy
    pair_result_merged_df["accuracy"] = (
        (
            pair_result_merged_df[repeat_forward_cols]
            .eq(pair_result_merged_df["setup_true"], axis=0)
            .sum(axis=1)
            + pair_result_merged_df[repeat_reverse_cols]
            .ne(pair_result_merged_df["setup_true"], axis=0)
            .sum(axis=1)
        )
        / REPEAT_NUM
        / 2
    )

    # Bias (+1 means a tendency to choose the first item, -1 means a tendency to choose the second item) *Note that for repeat_reverse_cols, B is originally A
    pair_result_merged_df["first_item_bias"] = (
        (pair_result_merged_df[repeat_forward_cols] == "A").sum(axis=1)
        + (pair_result_merged_df[repeat_reverse_cols] == "A").sum(axis=1)
    ) / REPEAT_NUM - 1

    # Preference (+1 means a tendency to choose A, -1 means a tendency to choose B) *Note that for repeat_reverse_cols, B is originally A
    pair_result_merged_df["A_preference"] = (
        (pair_result_merged_df[repeat_forward_cols] == "A").sum(axis=1)
        + (pair_result_merged_df[repeat_reverse_cols] == "B").sum(axis=1)
    ) / REPEAT_NUM - 1

    pair_result_dfs[(dataset_name, model_w_params)] = pair_result_merged_df


In [ ]:
scores = []
for (dataset_name, model_w_params), pair_result_df in pair_result_dfs.items():
    factor = t.ppf(1 - 0.05 / 2, df=len(pair_result_df) - 1) / np.sqrt(
        len(pair_result_df)
    )

    accuracy_mean = pair_result_df["accuracy"].mean()
    accuracy_ci_half_width = pair_result_df["accuracy"].std() * factor

    first_item_bias_mean = pair_result_df["first_item_bias"].mean()
    first_item_bias_ci_half_width = pair_result_df["first_item_bias"].std() * factor

    first_item_bias_abs_mean = pair_result_df["first_item_bias"].abs().mean()
    first_item_bias_abs_ci_half_width = (
        pair_result_df["first_item_bias"].abs().std() * factor
    )

    A_preference_mean = pair_result_df["A_preference"].mean()
    A_preference_ci_half_width = pair_result_df["A_preference"].std() * factor

    A_preference_abs_mean = pair_result_df["A_preference"].abs().mean()
    A_preference_abs_ci_half_width = pair_result_df["A_preference"].abs().std() * factor

    scores.append(
        {
            "dataset_name": dataset_name,
            "model_w_params": model_w_params,
            "accuracy_mean": accuracy_mean,
            "accuracy_ci_half_width": accuracy_ci_half_width,
            "first_item_bias_mean": first_item_bias_mean,
            "first_item_bias_ci_half_width": first_item_bias_ci_half_width,
            "first_item_bias_abs_mean": first_item_bias_abs_mean,
            "first_item_bias_abs_ci_half_width": first_item_bias_abs_ci_half_width,
            "A_preference_mean": A_preference_mean,
            "A_preference_ci_half_width": A_preference_ci_half_width,
            "A_preference_abs_mean": A_preference_abs_mean,
            "A_preference_abs_ci_half_width": A_preference_abs_ci_half_width,
        }
    )
scores_df = pd.DataFrame(scores)

# Save
scores_df.to_csv("results_final/scores.csv", index=False)

In [ ]:
# Plot bias and preference with error bars
vars = [
    "accuracy_mean",
    "first_item_bias_mean",
    "A_preference_abs_mean",
]
var_labels = {
    "accuracy_mean": "Mean accuracy",
    "first_item_bias_mean": "Mean position bias",
    "A_preference_abs_mean": "Mean preference consistency",
}
ncol = 3
nsub = len(vars)
nrow = nsub // ncol + (nsub % ncol > 0)
plt.figure(figsize=(ncol * 5, nrow * 4))
handles, labels = None, None
for i, var in enumerate(vars):
    ax = plt.subplot(nrow, ncol, i + 1)

    x_labels = scores_df["dataset_name"].unique()
    models = scores_df["model_w_params"].unique()
    x = np.arange(len(x_labels))
    width = 0.8 / len(models)

    for j, model in enumerate(models):
        model_data = scores_df[scores_df["model_w_params"] == model]
        means = model_data[var].to_numpy()
        cis = model_data[var.replace("_mean", "_ci_half_width")].to_numpy()

        ax.bar(
            x + j * width,
            means,
            width=width,
            label=model,
            yerr=cis,
            capsize=3 if cis is not None else 0,
            alpha=0.8,
        )
    if var == "accuracy_mean":
        ax.set_ylim(0.0, 1.0)
        ax.set_yticks(np.arange(0.0, 1.01, 0.1))
    elif var == "first_item_bias_mean":
        ax.set_ylim(-1.0, 1.0)
        ax.set_yticks(np.arange(-1.0, 1.01, 0.2))
        ax.axhline(0, color="black", linewidth=0.5)
    elif var == "A_preference_abs_mean":
        ax.set_ylim(0.0, 1.0)
        ax.set_yticks(np.arange(0.0, 1.01, 0.1))
    ax.grid(axis="y", linestyle="--", alpha=0.3)
    ax.tick_params(axis="y", labelsize=10)
    ax.set_ylabel(f"{var_labels[var]}", fontsize=12)
    ax.set_xticks(x + width * (len(models) - 1) / 2)
    ax.set_xticklabels(x_labels, fontsize=10, rotation=90)
    ax.set_xlabel("", fontsize=12)
    if i == 0:
        handles, labels = ax.get_legend_handles_labels()
    leg = ax.get_legend()
    if leg is not None:
        leg.remove()
plt.tight_layout()
plt.figlegend(
    handles,
    labels,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.2),
    fontsize=10,
    frameon=True,
    ncol=len(models) // 3,
)
plt.savefig(
    "images/accuracy_position_bias_preference_consistency_horizontal.png",
    format="png",
    dpi=600,
    bbox_inches="tight",
)
plt.savefig(
    "images/accuracy_position_bias_preference_consistency_horizontal.pdf",
    format="pdf",
    bbox_inches="tight",
)
plt.savefig(
    "images/accuracy_position_bias_preference_consistency_horizontal.svg",
    format="svg",
    bbox_inches="tight",
)
plt.show()

In [ ]:
# Plot bias and preference with error bars
vars = [
    "accuracy_mean",
    "first_item_bias_mean",
    "A_preference_abs_mean",
]
var_labels = {
    "accuracy_mean": "Mean accuracy",
    "first_item_bias_mean": "Mean position bias",
    "A_preference_abs_mean": "Mean preference consistency",
}
ncol = 1
nsub = len(vars)
nrow = nsub // ncol + (nsub % ncol > 0)
fig, axes = plt.subplots(nrow, ncol, figsize=(ncol * 5, nrow * 3), sharex=True)
axes = np.atleast_1d(axes).ravel()
handles, labels = None, None
for i, var in enumerate(vars):
    ax = plt.subplot(nrow, ncol, i + 1)

    x_labels = scores_df["dataset_name"].unique()
    models = scores_df["model_w_params"].unique()
    x = np.arange(len(x_labels))
    width = 0.8 / len(models)

    for j, model in enumerate(models):
        model_data = scores_df[scores_df["model_w_params"] == model]
        means = model_data[var].to_numpy()
        cis = model_data[var.replace("_mean", "_ci_half_width")].to_numpy()
        ax.bar(
            x + j * width,
            means,
            width=width,
            label=model,
            yerr=cis,
            capsize=3 if cis is not None else 0,
            alpha=0.8,
        )
    if var == "accuracy_mean":
        ax.set_ylim(0.0, 1.0)
        ax.set_yticks(np.arange(0.0, 1.01, 0.1))
    elif var == "first_item_bias_mean":
        ax.set_ylim(-1.0, 1.0)
        ax.set_yticks(np.arange(-1.0, 1.01, 0.2))
        ax.axhline(0, color="black", linewidth=0.5)
    elif var == "A_preference_abs_mean":
        ax.set_ylim(0.0, 1.0)
        ax.set_yticks(np.arange(0.0, 1.01, 0.1))
    ax.grid(axis="y", linestyle="--", alpha=0.3)
    ax.tick_params(axis="y", labelsize=10)
    ax.set_ylabel(f"{var_labels[var]}", fontsize=12)
    ax.set_xticks(x + width * (len(models) - 1) / 2)
    ax.set_xticklabels(x_labels, fontsize=10, rotation=90)
    ax.set_xlabel("", fontsize=12)
    if i == 0:
        handles, labels = ax.get_legend_handles_labels()
    leg = ax.get_legend()
    if leg is not None:
        leg.remove()
plt.figlegend(
    handles,
    labels,
    loc="center right",
    bbox_to_anchor=(1.5, 0.524),
    fontsize=10,
    frameon=True,
    ncol=1,
)
plt.tight_layout()
plt.savefig(
    "images/accuracy_position_bias_preference_consistency_vertical.png",
    format="png",
    dpi=600,
    bbox_inches="tight",
)
plt.savefig(
    "images/accuracy_position_bias_preference_consistency_vertical.pdf",
    format="pdf",
    bbox_inches="tight",
)
plt.savefig(
    "images/accuracy_position_bias_preference_consistency_vertical.svg",
    format="svg",
    bbox_inches="tight",
)
plt.show()

## Frequency distribution

In [ ]:
MODEL_W_PARAMS_ORDER = [
    "GPT-5 mini medium",
    "GPT-4.1 mini temp1.0",
    "GPT-4o mini temp1.0",
    "Claude Sonnet 4.5 temp1.0",
    "Claude Haiku 4.5 temp1.0",
    "Claude 3.5 Haiku temp1.0",
]
sorted_keys = sorted(
    pair_result_dfs.keys(),
    key=lambda x: (
        MODEL_W_PARAMS_ORDER.index(x[1])
        if x[1] in MODEL_W_PARAMS_ORDER
        else len(MODEL_W_PARAMS_ORDER),
        DATASET_NAME_ORDER.index(x[0])
        if x[0] in DATASET_NAME_ORDER
        else len(DATASET_NAME_ORDER),
    ),
)
pair_result_dfs_sorted = {key: pair_result_dfs[key] for key in sorted_keys}

In [ ]:
# Plot of bias vs. preference
ncol = len(np.unique([model_w_params for _, model_w_params in pair_result_dfs]))
nsub = len(pair_result_dfs)
nrow = nsub // ncol + (nsub % ncol > 0)
plt.figure(figsize=(ncol * 4, nrow * 3))
for i, ((dataset_name, model_w_params), pair_result_df) in enumerate(
    pair_result_dfs.items()
):
    plt.subplot(nrow, ncol, i + 1)
    x_bins = np.arange(-1.1, 1.2, 0.2)
    y_bins = np.arange(-1.1, 1.2, 0.2)
    freq_table, xedges, yedges = np.histogram2d(
        pair_result_df["first_item_bias"],
        pair_result_df["A_preference"],
        bins=[x_bins, y_bins],
    )
    freq_table = freq_table.astype(int)
    x_labels = [round(x + 0.1, 1) for x in x_bins[:-1]]
    y_labels = [round(y + 0.1, 1) for y in y_bins[:-1]]
    ax = sns.heatmap(
        freq_table,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=y_labels,
        yticklabels=x_labels,
        annot_kws={"size": 8},
    )
    ax.collections[0].colorbar.ax.tick_params(labelsize=8)
    plt.xticks(fontsize=8)
    plt.yticks(fontsize=8)
    plt.xlabel("Condition preference", fontsize=10)
    plt.ylabel("Position bias", fontsize=10)
    plt.title(f"{dataset_name} - {model_w_params}", fontsize=10)
plt.tight_layout()
plt.savefig(
    "images/condition_preference_position_bias_horizontal.png",
    format="png",
    dpi=300,
    bbox_inches="tight",
)
plt.savefig(
    "images/condition_preference_position_bias_horizontal.pdf",
    format="pdf",
    bbox_inches="tight",
)
plt.savefig(
    "images/condition_preference_position_bias_horizontal.svg",
    format="svg",
    bbox_inches="tight",
)
plt.show()

In [ ]:
# Plot of bias vs. preference
nrow = len(np.unique([model_w_params for _, model_w_params in pair_result_dfs_sorted]))
nsub = len(pair_result_dfs_sorted)
ncol = nsub // nrow + (nsub % nrow > 0)
plt.figure(figsize=(ncol * 4, nrow * 3))
for i, ((dataset_name, model_w_params), pair_result_df) in enumerate(
    pair_result_dfs_sorted.items()
):
    plt.subplot(nrow, ncol, i + 1)
    x_bins = np.arange(-1.1, 1.2, 0.2)
    y_bins = np.arange(-1.1, 1.2, 0.2)
    freq_table, xedges, yedges = np.histogram2d(
        pair_result_df["first_item_bias"],
        pair_result_df["A_preference"],
        bins=[x_bins, y_bins],
    )
    freq_table = freq_table.astype(int)
    x_labels = [round(x + 0.1, 1) for x in x_bins[:-1]]
    y_labels = [round(y + 0.1, 1) for y in y_bins[:-1]]
    ax = sns.heatmap(
        freq_table,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=y_labels,
        yticklabels=x_labels,
        annot_kws={"size": 8},
    )
    ax.collections[0].colorbar.ax.tick_params(labelsize=8)
    plt.xticks(fontsize=8)
    plt.yticks(fontsize=8)
    plt.xlabel("Condition preference", fontsize=10)
    plt.ylabel("Position bias", fontsize=10)
    plt.title(f"{dataset_name} - {model_w_params}", fontsize=10)
plt.tight_layout()
plt.savefig(
    "images/condition_preference_position_bias_vertical.png",
    format="png",
    dpi=300,
    bbox_inches="tight",
)
plt.savefig(
    "images/condition_preference_position_bias_vertical.pdf",
    format="pdf",
    bbox_inches="tight",
)
plt.savefig(
    "images/condition_preference_position_bias_vertical.svg",
    format="svg",
    bbox_inches="tight",
)
plt.show()

In [ ]:
# Plot of accuracy vs. preference
ncol = len(np.unique([model_w_params for _, model_w_params in pair_result_dfs]))
nsub = len(pair_result_dfs)
nrow = nsub // ncol + (nsub % ncol > 0)
plt.figure(figsize=(ncol * 4, nrow * 3))
for i, ((dataset_name, model_w_params), pair_result_df) in enumerate(
    pair_result_dfs.items()
):
    plt.subplot(nrow, ncol, i + 1)
    x_bins = np.arange(-0.05, 1.1, 0.1)
    y_bins = np.arange(-1.1, 1.2, 0.2)
    freq_table, xedges, yedges = np.histogram2d(
        pair_result_df["accuracy"],
        pair_result_df["A_preference"],
        bins=[x_bins, y_bins],
    )
    freq_table = freq_table.astype(int)
    x_labels = [round(x + 0.05, 1) for x in x_bins[:-1]]
    y_labels = [round(y + 0.1, 1) for y in y_bins[:-1]]
    ax = sns.heatmap(
        freq_table,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=y_labels,
        yticklabels=x_labels,
        annot_kws={"size": 8},
    )
    ax.collections[0].colorbar.ax.tick_params(labelsize=8)
    plt.xticks(fontsize=8)
    plt.yticks(fontsize=8)
    plt.xlabel("Condition preference", fontsize=10)
    plt.ylabel("Accuracy", fontsize=10)
    plt.title(f"{dataset_name} - {model_w_params}", fontsize=10)
plt.tight_layout()
plt.savefig(
    "images/condition_preference_accuracy_horizontal.png",
    format="png",
    dpi=300,
    bbox_inches="tight",
)
plt.savefig(
    "images/condition_preference_accuracy_horizontal.pdf",
    format="pdf",
    bbox_inches="tight",
)
plt.savefig(
    "images/condition_preference_accuracy_horizontal.svg",
    format="svg",
    bbox_inches="tight",
)
plt.show()

In [ ]:
# Plot of accuracy vs. preference
nrow = len(np.unique([model_w_params for _, model_w_params in pair_result_dfs_sorted]))
nsub = len(pair_result_dfs_sorted)
ncol = nsub // nrow + (nsub % nrow > 0)
plt.figure(figsize=(ncol * 4, nrow * 3))
for i, ((dataset_name, model_w_params), pair_result_df) in enumerate(
    pair_result_dfs_sorted.items()
):
    plt.subplot(nrow, ncol, i + 1)
    x_bins = np.arange(-0.05, 1.1, 0.1)
    y_bins = np.arange(-1.1, 1.2, 0.2)
    freq_table, xedges, yedges = np.histogram2d(
        pair_result_df["accuracy"],
        pair_result_df["A_preference"],
        bins=[x_bins, y_bins],
    )
    freq_table = freq_table.astype(int)
    x_labels = [round(x + 0.05, 1) for x in x_bins[:-1]]
    y_labels = [round(y + 0.1, 1) for y in y_bins[:-1]]
    ax = sns.heatmap(
        freq_table,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=y_labels,
        yticklabels=x_labels,
        annot_kws={"size": 8},
    )
    ax.collections[0].colorbar.ax.tick_params(labelsize=8)
    plt.xticks(fontsize=8)
    plt.yticks(fontsize=8)
    plt.xlabel("Condition preference", fontsize=10)
    plt.ylabel("Accuracy", fontsize=10)
    plt.title(f"{dataset_name} - {model_w_params}", fontsize=10)
plt.tight_layout()
plt.savefig(
    "images/condition_preference_accuracy_vertical.png",
    format="png",
    dpi=300,
    bbox_inches="tight",
)
plt.savefig(
    "images/condition_preference_accuracy_vertical.pdf",
    format="pdf",
    bbox_inches="tight",
)
plt.savefig(
    "images/condition_preference_accuracy_vertical.svg",
    format="svg",
    bbox_inches="tight",
)
plt.show()

In [ ]:
# Plot of accuracy vs. bias
ncol = len(np.unique([model_w_params for _, model_w_params in pair_result_dfs]))
nsub = len(pair_result_dfs)
nrow = nsub // ncol + (nsub % ncol > 0)
plt.figure(figsize=(ncol * 4, nrow * 3))
for i, ((dataset_name, model_w_params), pair_result_df) in enumerate(
    pair_result_dfs.items()
):
    plt.subplot(nrow, ncol, i + 1)
    x_bins = np.arange(-0.05, 1.1, 0.1)
    y_bins = np.arange(-1.1, 1.2, 0.2)
    freq_table, xedges, yedges = np.histogram2d(
        pair_result_df["accuracy"],
        pair_result_df["first_item_bias"],
        bins=[x_bins, y_bins],
    )
    freq_table = freq_table.astype(int)
    x_labels = [round(x + 0.05, 1) for x in x_bins[:-1]]
    y_labels = [round(y + 0.1, 1) for y in y_bins[:-1]]
    ax = sns.heatmap(
        freq_table,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=y_labels,
        yticklabels=x_labels,
        annot_kws={"size": 8},
    )
    ax.collections[0].colorbar.ax.tick_params(labelsize=8)
    plt.xticks(fontsize=8)
    plt.yticks(fontsize=8)
    plt.xlabel("Position bias", fontsize=10)
    plt.ylabel("Accuracy", fontsize=10)
    plt.title(f"{dataset_name} - {model_w_params}", fontsize=10)
plt.tight_layout()
plt.savefig(
    "images/position_bias_accuracy_horizontal.png",
    format="png",
    dpi=300,
    bbox_inches="tight",
)
plt.savefig(
    "images/position_bias_accuracy_horizontal.pdf", format="pdf", bbox_inches="tight"
)
plt.savefig(
    "images/position_bias_accuracy_horizontal.svg", format="svg", bbox_inches="tight"
)
plt.show()

In [ ]:
# Plot of accuracy vs. bias
nrow = len(np.unique([model_w_params for _, model_w_params in pair_result_dfs_sorted]))
nsub = len(pair_result_dfs_sorted)
ncol = nsub // nrow + (nsub % nrow > 0)
plt.figure(figsize=(ncol * 4, nrow * 3))
for i, ((dataset_name, model_w_params), pair_result_df) in enumerate(
    pair_result_dfs_sorted.items()
):
    plt.subplot(nrow, ncol, i + 1)
    x_bins = np.arange(-0.05, 1.1, 0.1)
    y_bins = np.arange(-1.1, 1.2, 0.2)
    freq_table, xedges, yedges = np.histogram2d(
        pair_result_df["accuracy"],
        pair_result_df["first_item_bias"],
        bins=[x_bins, y_bins],
    )
    freq_table = freq_table.astype(int)
    x_labels = [round(x + 0.05, 1) for x in x_bins[:-1]]
    y_labels = [round(y + 0.1, 1) for y in y_bins[:-1]]
    ax = sns.heatmap(
        freq_table,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=y_labels,
        yticklabels=x_labels,
        annot_kws={"size": 8},
    )
    ax.collections[0].colorbar.ax.tick_params(labelsize=8)
    plt.xticks(fontsize=8)
    plt.yticks(fontsize=8)
    plt.xlabel("Position bias", fontsize=10)
    plt.ylabel("Accuracy", fontsize=10)
    plt.title(f"{dataset_name} - {model_w_params}", fontsize=10)
plt.tight_layout()
plt.savefig(
    "images/position_bias_accuracy_vertical.png",
    format="png",
    dpi=300,
    bbox_inches="tight",
)
plt.savefig(
    "images/position_bias_accuracy_vertical.pdf", format="pdf", bbox_inches="tight"
)
plt.savefig(
    "images/position_bias_accuracy_vertical.svg", format="svg", bbox_inches="tight"
)
plt.show()

## Targets

In [ ]:
# Scatter plot of accuracy vs. obj_var_diff
ncol = len(np.unique([model_w_params for _, model_w_params in pair_result_dfs]))
nsub = len(pair_result_dfs)
nrow = nsub // ncol + (nsub % ncol > 0)
plt.figure(figsize=(ncol * 3, nrow * 3))
for i, ((dataset_name, model_w_params), pair_result_df) in enumerate(
    pair_result_dfs.items()
):
    obj_var = DATASET_VARS[dataset_name]["obj_var"]
    plt.subplot(nrow, ncol, i + 1)
    sns.scatterplot(
        data=pair_result_df,
        x=f"{obj_var}_diff",
        y="accuracy",
        alpha=0.5,
    )
    plt.ylim(-0.1, 1.1)
    plt.xticks(fontsize=8)
    plt.yticks(fontsize=8)
    plt.xlabel(f"{obj_var} difference", fontsize=10)
    plt.ylabel("Accuracy", fontsize=10)
    plt.title(f"{dataset_name} - {model_w_params}", fontsize=10)
plt.tight_layout()
# plt.savefig("images/accuracy_target_diff_horizontal.png", format="png", dpi=300, bbox_inches="tight")
# plt.savefig("images/accuracy_target_diff_horizontal.pdf", format="pdf", bbox_inches="tight")
# plt.savefig("images/accuracy_target_diff_horizontal.svg", format="svg", bbox_inches="tight")
plt.show()

In [ ]:
# Scatter plot of accuracy vs. obj_var_diff
nrow = len(np.unique([model_w_params for _, model_w_params in pair_result_dfs_sorted]))
nsub = len(pair_result_dfs_sorted)
ncol = nsub // nrow + (nsub % nrow > 0)
plt.figure(figsize=(ncol * 3, nrow * 3))
for i, ((dataset_name, model_w_params), pair_result_df) in enumerate(
    pair_result_dfs_sorted.items()
):
    obj_var = DATASET_VARS[dataset_name]["obj_var"]
    plt.subplot(nrow, ncol, i + 1)
    sns.scatterplot(
        data=pair_result_df,
        x=f"{obj_var}_diff",
        y="accuracy",
        alpha=0.5,
    )
    plt.ylim(-0.1, 1.1)
    plt.xticks(fontsize=8)
    plt.yticks(fontsize=8)
    plt.xlabel(f"{obj_var} difference", fontsize=10)
    plt.ylabel("Accuracy", fontsize=10)
    plt.title(f"{dataset_name} - {model_w_params}", fontsize=10)
plt.tight_layout()
plt.savefig(
    "images/accuracy_target_diff_vertical.png",
    format="png",
    dpi=300,
    bbox_inches="tight",
)
plt.savefig(
    "images/accuracy_target_diff_vertical.pdf", format="pdf", bbox_inches="tight"
)
plt.savefig(
    "images/accuracy_target_diff_vertical.svg", format="svg", bbox_inches="tight"
)
plt.show()

In [ ]:
# Scatter plot of bias vs. obj_var_diff
ncol = len(np.unique([model_w_params for _, model_w_params in pair_result_dfs]))
nsub = len(pair_result_dfs)
nrow = nsub // ncol + (nsub % ncol > 0)
plt.figure(figsize=(ncol * 3, nrow * 3))
for i, ((dataset_name, model_w_params), pair_result_df) in enumerate(
    pair_result_dfs.items()
):
    obj_var = DATASET_VARS[dataset_name]["obj_var"]
    plt.subplot(nrow, ncol, i + 1)
    sns.scatterplot(
        data=pair_result_df,
        x=f"{obj_var}_diff",
        y="first_item_bias",
        alpha=0.5,
    )
    plt.ylim(-1.1, 1.1)
    plt.xticks(fontsize=8)
    plt.yticks(fontsize=8)
    plt.xlabel(f"{obj_var} difference", fontsize=10)
    plt.ylabel("Position bias", fontsize=10)
    plt.title(f"{dataset_name} - {model_w_params}", fontsize=10)
plt.tight_layout()
plt.savefig(
    "images/position_bias_target_diff_horizontal.png",
    format="png",
    dpi=300,
    bbox_inches="tight",
)
plt.savefig(
    "images/position_bias_target_diff_horizontal.pdf", format="pdf", bbox_inches="tight"
)
plt.savefig(
    "images/position_bias_target_diff_horizontal.svg", format="svg", bbox_inches="tight"
)
plt.show()

In [ ]:
# Scatter plot of bias vs. obj_var_diff
nrow = len(np.unique([model_w_params for _, model_w_params in pair_result_dfs_sorted]))
nsub = len(pair_result_dfs_sorted)
ncol = nsub // nrow + (nsub % nrow > 0)
plt.figure(figsize=(ncol * 3, nrow * 3))
for i, ((dataset_name, model_w_params), pair_result_df) in enumerate(
    pair_result_dfs_sorted.items()
):
    obj_var = DATASET_VARS[dataset_name]["obj_var"]
    plt.subplot(nrow, ncol, i + 1)
    sns.scatterplot(
        data=pair_result_df,
        x=f"{obj_var}_diff",
        y="first_item_bias",
        alpha=0.5,
    )
    plt.ylim(-1.1, 1.1)
    plt.xticks(fontsize=8)
    plt.yticks(fontsize=8)
    plt.xlabel(f"{obj_var} difference", fontsize=10)
    plt.ylabel("Position bias", fontsize=10)
    plt.title(f"{dataset_name} - {model_w_params}", fontsize=10)
plt.tight_layout()
plt.savefig(
    "images/position_bias_target_diff_vertical.png",
    format="png",
    dpi=300,
    bbox_inches="tight",
)
plt.savefig(
    "images/position_bias_target_diff_vertical.pdf", format="pdf", bbox_inches="tight"
)
plt.savefig(
    "images/position_bias_target_diff_vertical.svg", format="svg", bbox_inches="tight"
)
plt.show()

In [ ]:
# Scatter plot of preference vs. obj_var_diff
ncol = len(np.unique([model_w_params for _, model_w_params in pair_result_dfs]))
nsub = len(pair_result_dfs)
nrow = nsub // ncol + (nsub % ncol > 0)
plt.figure(figsize=(ncol * 3, nrow * 3))
for i, ((dataset_name, model_w_params), pair_result_df) in enumerate(
    pair_result_dfs.items()
):
    obj_var = DATASET_VARS[dataset_name]["obj_var"]
    plt.subplot(nrow, ncol, i + 1)
    sns.scatterplot(
        data=pair_result_df,
        x=f"{obj_var}_diff",
        y="A_preference",
        alpha=0.5,
    )
    plt.ylim(-1.1, 1.1)
    plt.xticks(fontsize=8)
    plt.yticks(fontsize=8)
    plt.xlabel(f"{obj_var} difference", fontsize=10)
    plt.ylabel("Condition preference", fontsize=10)
    plt.title(f"{dataset_name} - {model_w_params}", fontsize=10)
plt.tight_layout()
plt.savefig(
    "images/condition_preference_target_diff_horizontal.png",
    format="png",
    dpi=300,
    bbox_inches="tight",
)
plt.savefig(
    "images/condition_preference_target_diff_horizontal.pdf",
    format="pdf",
    bbox_inches="tight",
)
plt.savefig(
    "images/condition_preference_target_diff_horizontal.svg",
    format="svg",
    bbox_inches="tight",
)
plt.show()

In [ ]:
# Scatter plot of preference vs. obj_var_diff
nrow = len(np.unique([model_w_params for _, model_w_params in pair_result_dfs_sorted]))
nsub = len(pair_result_dfs_sorted)
ncol = nsub // nrow + (nsub % nrow > 0)
plt.figure(figsize=(ncol * 3, nrow * 3))
for i, ((dataset_name, model_w_params), pair_result_df) in enumerate(
    pair_result_dfs_sorted.items()
):
    obj_var = DATASET_VARS[dataset_name]["obj_var"]
    plt.subplot(nrow, ncol, i + 1)
    sns.scatterplot(
        data=pair_result_df,
        x=f"{obj_var}_diff",
        y="A_preference",
        alpha=0.5,
    )
    plt.ylim(-1.1, 1.1)
    plt.xticks(fontsize=8)
    plt.yticks(fontsize=8)
    plt.xlabel(f"{obj_var} difference", fontsize=10)
    plt.ylabel("Condition preference", fontsize=10)
    plt.title(f"{dataset_name} - {model_w_params}", fontsize=10)
plt.tight_layout()
plt.savefig(
    "images/condition_preference_target_diff_vertical.png",
    format="png",
    dpi=300,
    bbox_inches="tight",
)
plt.savefig(
    "images/condition_preference_target_diff_vertical.pdf",
    format="pdf",
    bbox_inches="tight",
)
plt.savefig(
    "images/condition_preference_target_diff_vertical.svg",
    format="svg",
    bbox_inches="tight",
)
plt.show()